# Hospital Resource Allocation Optimization
## Final Report — GWO vs PSO Comparison

**Project:** Hospital Resource Allocation Optimization using Metaheuristics
**Method:** Grey Wolf Optimizer (GWO) vs Particle Swarm Optimization (PSO)
**Datasets:** 5 CSV files (patient, staff, inventory, financial, vendor)
**Runs:** 30 independent runs per algorithm

### Executive Summary

This report presents the results of optimizing hospital resource allocation using two
nature-inspired metaheuristic algorithms. The Grey Wolf Optimizer (GWO) consistently
outperforms Particle Swarm Optimization (PSO) in terms of solution quality (lower total
cost), convergence speed, and statistical robustness, as demonstrated by non-parametric
statistical tests (Wilcoxon p < 0.05).


## 1. Data Quality Findings

Five raw CSV datasets were audited before preprocessing. Key issues discovered:

| Dataset | Rows | Key Issues |
|---|---|---|
| patient_data | 351 | ~28% records had Discharge_Date < Admission_Date; Staff_Needed formatting inconsistent |
| inventory_data | 502 | Current_Stock > Max_Capacity in multiple rows; Min_Required > Max_Capacity violations |
| staff_data | 502 | Shift Hours_Worked inconsistent with computed shift duration (cross-midnight shifts) |
| financial_data | 502 | Description category mismatches (e.g., "Surgical masks" filed under "Staffing") |
| vendor_data | 4 | Very sparse (3 vendors); single item per vendor |

### Preprocessing Actions

- **patient**: Swapped inverted date pairs; recomputed Bed_Days; parsed Staff_Needed into typed columns
- **inventory**: Capped stock/min violations; IQR-clipped Unit_Cost; derived Days_Of_Stock, Restock_Urgency
- **staff**: Computed actual shift hours handling cross-midnight; corrected Hours_Worked; recomputed Overtime_Hours
- **financial**: Inferred correct category from Description keywords; flagged mismatches
- **vendor**: Parsed dates; computed Actual_Lead_Time


## 2. Exploratory Data Analysis

![Monthly Expense by Category (corrected)](figures/financial_trends.png)

*Monthly Expense by Category (corrected)*

![Expense Amount Distribution by Category](figures/financial_amount_dist.png)

*Expense Amount Distribution by Category*

![Inventory Stockout Risk Heatmap (Item × Month)](figures/inventory_stockout_risk.png)

*Inventory Stockout Risk Heatmap (Item × Month)*

![Staff Utilization Rate by Type](figures/staff_utilization.png)

*Staff Utilization Rate by Type*

![Patient Primary Diagnosis Distribution](figures/patient_diagnosis_mix.png)

*Patient Primary Diagnosis Distribution*

![Bed Days Distribution by Room Type](figures/patient_bed_days_dist.png)

*Bed Days Distribution by Room Type*

## 3. Optimization Problem Formulation

### Decision Variables

| Variable | Description | Dimension |
|---|---|---|
| Staff allocation | Number of each staff type per shift | n_shifts × 3 |
| Inventory restock | Order quantity per SKU | n_inventory_items |

**Total decision variable dimension ≈ 100** (30 shifts × 3 staff types + 10 inventory items)

### Objective Function

$$\text{Fitness} = \underbrace{C_{\text{staff}} + C_{\text{overtime}} + C_{\text{restock}} + C_{\text{holding}}}_{\text{total cost}} + \lambda \sum_{j} v_j$$

Where $v_j$ are constraint violation magnitudes and $\lambda = 10^6$ (penalty weight).

### Constraints (via Penalty Method)

1. **Minimum staffing**: Each shift must have ≥ minimum required staff per type
2. **Budget**: Daily staffing cost ≤ daily budget limit
3. **Inventory safety stock**: Order quantity ≥ minimum required order per SKU


## 4. Algorithm Descriptions

### Grey Wolf Optimizer (GWO) — Mirjalili et al., 2014

GWO simulates the leadership hierarchy and hunting behaviour of grey wolves.
Wolves are ranked α (best), β (second), δ (third), and ω (remaining).
Each ω wolf updates its position as the **mean of three leader-guided movements**:

$$\vec{X}(t+1) = \frac{\vec{X}_1 + \vec{X}_2 + \vec{X}_3}{3}$$

The coefficient $a$ decreases linearly from 2 to 0, controlling the balance between
exploration ($|A| > 1$) and exploitation ($|A| < 1$).

### Particle Swarm Optimization (PSO) — Kennedy & Eberhart, 1995

PSO models social behaviour of bird flocking. Each particle updates velocity as:

$$v_i(t+1) = w \cdot v_i(t) + c_1 r_1 (p_{\text{best},i} - x_i) + c_2 r_2 (g_{\text{best}} - x_i)$$

Inertia weight $w$ decays linearly from 0.9 to 0.4. $c_1 = c_2 = 2.0$.

### Why GWO Outperforms PSO on This Problem

1. **Three-leader exploitation**: α/β/δ hierarchy provides diverse search directions toward feasible regions; PSO's single global best concentrates particles prematurely
2. **Adaptive balance**: GWO's single decaying parameter $a$ is more stable than PSO's dual-coefficient tuning on tightly-constrained problems
3. **Constraint handling**: GWO's bounded stochastic walks naturally maintain proximity to feasible regions; PSO oscillates near penalty boundaries
4. **High-dimensional robustness**: On ~100-dim problems, GWO's coordinate-averaging mitigates the curse of dimensionality better than PSO's velocity momentum


## 5. Experiment Results (30 Independent Runs)

In [ ]:
import pandas as pd
summary_data = [{'Metric': 'Mean Fitness', 'GWO': '120090.9278', 'PSO': '219360.0000'}, {'Metric': 'Std Fitness', 'GWO': '1171.4436', 'PSO': '27399.9182'}, {'Metric': 'Min Fitness', 'GWO': '118059.3663', 'PSO': '169320.0000'}, {'Metric': 'Median Fitness', 'GWO': '120183.7632', 'PSO': '216300.0000'}, {'Metric': 'Mean Runtime (s)', 'GWO': '1.192', 'PSO': '0.638'}, {'Metric': 'Wilcoxon p-value', 'GWO': '9.3132e-10', 'PSO': '—'}, {'Metric': 'Wilcoxon significant', 'GWO': 'True', 'PSO': '—'}, {'Metric': 'Mann-Whitney p-value', 'GWO': '1.5090e-11', 'PSO': '—'}, {'Metric': 'Mann-Whitney significant', 'GWO': 'True', 'PSO': '—'}, {'Metric': 'Effect size r (rank-biserial)', 'GWO': '1.0000', 'PSO': '—'}]
df = pd.DataFrame(summary_data)
df.style.set_caption("GWO vs PSO Statistical Comparison")

## 6. Convergence Analysis

![Convergence Curves: Mean ± Std across 30 Runs](figures/convergence_curves.png)

*Convergence Curves: Mean ± Std across 30 Runs*

GWO converges faster and to a lower fitness value compared to PSO.
The shaded regions show ±1 standard deviation — GWO also exhibits lower variance,
indicating more consistent performance across random initialisations.

## 7. Statistical Test Results

The following non-parametric tests confirm GWO's superiority:

- **Wilcoxon Signed-Rank Test** (paired, one-sided H₁: GWO < PSO): Tests whether GWO's fitness values are systematically lower than PSO's. A significant result (p < 0.05) rejects the null hypothesis of equal distributions.
- **Mann-Whitney U Test** (unpaired, robustness check): Confirms the result without assuming pairing.
- **Effect Size** (rank-biserial correlation r): Measures practical significance. r > 0.5 indicates large effect.

See summary table above for exact p-values and effect size.


## 8. Best GWO Resource Allocation

![Best GWO Staff Allocation Matrix (Shifts × Staff Types)](figures/best_allocation_heatmap.png)

*Best GWO Staff Allocation Matrix (Shifts × Staff Types)*

## 9. Conclusion

### Key Findings

1. **GWO achieves lower total resource allocation cost** than PSO across all 30 independent runs
2. **Wilcoxon test confirms statistical significance** (p < 0.05), ruling out random variation
3. **Large effect size** (|r| > 0.5) indicates practical significance, not just statistical significance
4. **GWO converges faster**: reaches near-optimal solutions in fewer iterations
5. **GWO is more stable**: lower fitness variance across runs

### Why GWO is Recommended for Hospital Resource Allocation

The hospital resource allocation problem is a **high-dimensional, multi-constraint** optimization problem.
GWO's three-leader social hierarchy naturally handles such problems by maintaining multiple
high-quality reference points (α, β, δ) that guide exploration toward feasible regions.
PSO's single global best tends to premature convergence when penalty barriers are steep,
leading to suboptimal feasible solutions.

### Business Impact

The optimal GWO allocation reduces total operational costs (staffing + inventory + holding)
while satisfying all clinical constraints (minimum staffing levels, inventory safety stock,
budget compliance). The savings compared to PSO's solution scale linearly with the number
of shifts and SKUs.

---
*Report generated by the Hospital Resource Allocation Optimization pipeline.*
*Algorithms: GWO (Mirjalili et al., 2014) | PSO (Kennedy & Eberhart, 1995)*
